# 🌍 Notebook 00 — Data Acquisition & First Look

## Project: Climate Change & Economic Impact in Germany — A Berlin Close-Up

### Purpose of this notebook
This is the **reconnaissance notebook**. The goal is NOT to clean or analyse anything yet.  
The goal is to download raw data from each source, open it, and understand what we have.

Every dataset gets:
- A download (saved to `data/raw/`)
- A first look (shape, columns, date range)
- A short note on what we observe

### Research Questions this project will answer
- **RQ1** — Is climate change in Germany measurably accelerating?
- **RQ2** — Which economic sectors show the strongest correlation with climate damage?
- **RQ3** — Can we model economic damage from climate variables?
- **RQ4** — Is there an adaptation investment gap?
- **RQ5** — Is Berlin's heat and wildfire exposure worsening?

### Data sources I will check 
1. **DWD** (Deutscher Wetterdienst) — temperature & extreme weather time series
2. **GDV** (Gesamtverband der Deutschen Versicherungswirtschaft) — insured natural hazard damage
3. **LFB** (Landesbetrieb Forst Brandenburg) — Brandenburg wildfire statistics
4. **UBA** (Umweltbundesamt) — Germany climate indicators & adaptation monitoring
5. **KfW** — green & adaptation investment data
6. **Berliner Klimaatlas** — Berlin urban heat island geodata
7. **EFFIS** (European Forest Fire Information System) — EU-level wildfire data for cross-reference

### Realistic expectation
Some sources will give me clean downloadable files immediately.  
Some will be PDFs I need to extract tables from later.  
Some may require manual download from a web portal.  
All of that is useful information — knowing WHERE the friction is  
before we start cleaning is exactly the point of this notebook.

In [1]:
# ============================================================
# IMPORTS
# ============================================================
# These are all the libraries we need for this notebook.
# Think of libraries as toolboxes — each one gives us
# specific tools Python doesn't have built in.
# ============================================================

# pandas: the main tool for working with tables of data
# We'll use it to load, preview, and inspect every dataset
import pandas as pd

# requests: lets Python send requests to the internet
# Like a browser visiting a URL, but from inside Python
import requests

# zipfile: DWD packages their data as .zip files
# This library lets us unzip them without leaving Python
import zipfile

# io: lets us treat downloaded file content as if it were
# a file sitting on disk — avoids having to save/reload
import io

# os and pathlib: tools for navigating folders and file paths
# Path() is smarter than plain strings — works on Mac and Windows
import os
from pathlib import Path

# Tell pandas to always show all columns when previewing data
# Without this it hides columns with "..." when there are many
pd.set_option('display.max_columns', None)

# This just confirms everything loaded without errors
print("All libraries imported successfully ✓")

All libraries imported successfully ✓


In [16]:
# ============================================================
# PROJECT PATHS
# ============================================================
# Before we download anything we need to tell Python WHERE
# to save things. We define all folder paths here once,
# and then reuse these variables throughout the notebook.
# This way if you ever move the project folder, you only
# need to change ONE line — the PROJECT_ROOT.
# ============================================================

# Path.home() gives us the home directory of your Mac user
# e.g. /Users/erickburguenosalas
# We then navigate from there to your project folder
PROJECT_ROOT = Path.home() / "Desktop" / "capstone_climate_germany"

# These are the raw data subfolders — one per source
# We never modify files in raw/ — it's our original archive
DWD_RAW      = PROJECT_ROOT / "data" / "raw" / "dwd"
GDV_RAW      = PROJECT_ROOT / "data" / "raw" / "gdv"
LFB_RAW      = PROJECT_ROOT / "data" / "raw" / "lfb_brandenburg"
UBA_RAW      = PROJECT_ROOT / "data" / "raw" / "uba"
KFW_RAW      = PROJECT_ROOT / "data" / "raw" / "kfw"
BERLIN_RAW   = PROJECT_ROOT / "data" / "raw" / "berlin"
EFFIS_RAW    = PROJECT_ROOT / "data" / "raw" / "effis"
MUNICH_RE_RAW = PROJECT_ROOT / "data" / "raw" / "munich_re"
DESTATIS_RAW  = PROJECT_ROOT / "data" / "raw" / "destatis"

# This is where cleaned, analysis-ready files will go later
PROCESSED    = PROJECT_ROOT / "data" / "processed"

# This is where we'll save CSVs ready for Tableau
TABLEAU      = PROJECT_ROOT / "data" / "tableau"

# mkdir() creates the folder if it doesn't exist yet
# parents=True — also creates parent folders if needed
# exist_ok=True — no error if the folder already exists
all_folders = [
    DWD_RAW, GDV_RAW, LFB_RAW, UBA_RAW,
    KFW_RAW, BERLIN_RAW, EFFIS_RAW,
    PROCESSED, TABLEAU, MUNICH_RE_RAW, DESTATIS_RAW
]

for folder in all_folders:
    folder.mkdir(parents=True, exist_ok=True)

# Confirm everything was created
print("Project folder structure confirmed ✓")
print(f"\nRoot: {PROJECT_ROOT}")
print("\nSubfolders created:")
for folder in all_folders:
    # .relative_to() shows the path relative to root — cleaner to read
    print(f"  ✓ {folder.relative_to(PROJECT_ROOT)}")

Project folder structure confirmed ✓

Root: /Users/erickburguenosalas/Desktop/capstone_climate_germany

Subfolders created:
  ✓ data/raw/dwd
  ✓ data/raw/gdv
  ✓ data/raw/lfb_brandenburg
  ✓ data/raw/uba
  ✓ data/raw/kfw
  ✓ data/raw/berlin
  ✓ data/raw/effis
  ✓ data/processed
  ✓ data/tableau
  ✓ data/raw/munich_re
  ✓ data/raw/destatis


## 1. DWD — Deutscher Wetterdienst
Germany's national weather service. Clean, free, direct downloads.  
We start with the national annual mean temperature time series — our baseline for RQ1.

In [3]:
# ============================================================
# DWD — Annual mean temperature for Germany (1881–present)
# ============================================================
# DWD provides national averages as plain text files.
# This URL points directly to the file — no login needed.
# ============================================================

url = (
    "https://opendata.dwd.de/climate_environment/CDC/"
    "regional_averages_DE/annual/air_temperature_mean/"
    "regional_averages_tm_year.txt"
)

# Send a request to that URL and get the response
response = requests.get(url)

# Status code 200 means the server responded with "OK"
# Anything else means something went wrong
print(f"Status code: {response.status_code}")

# Show the first 1500 characters of the raw file
# so we can see how DWD formats it before we load it properly
print("\n--- RAW FILE PREVIEW ---\n")
print(response.text[:1500])

Status code: 200

--- RAW FILE PREVIEW ---

Zeitreihen fuer Gebietsmittel fuer Bundeslaender und Kombinationen von Bundeslaender, erstellt am: 20260402
Jahr;Jahr;Brandenburg/Berlin;Brandenburg;Baden-Wuerttemberg;Bayern;Hessen;Mecklenburg-Vorpommern;Niedersachsen;Niedersachsen/Hamburg/Bremen;Nordrhein-Westfalen;Rheinland-Pfalz;Schleswig-Holstein;Saarland;Sachsen;Sachsen-Anhalt;Thueringen/Sachsen-Anhalt;Thueringen;Deutschland;
1881;year;     7.55;     7.54;     7.66;     6.61;     7.49;     6.96;     7.54;     7.54;     8.14;     7.97;     7.12;     8.28;     6.71;     7.46;     7.11;     6.66;     7.31;
1882;year;     8.99;     8.97;     8.08;     7.33;     8.25;     8.54;     8.88;     8.88;     9.03;     8.55;     8.78;     8.79;     8.12;     8.81;     8.35;     7.77;     8.34;
1883;year;     8.42;     8.41;     7.77;     6.85;     7.96;     7.95;     8.39;     8.39;     8.71;     8.26;     8.18;     8.51;     7.46;     8.32;     7.87;     7.31;     7.88;
1884;year;     9.11;     9.1

### Load, clean column names and save raw file

In [4]:
# ============================================================
# Load the DWD temperature file into a pandas DataFrame
# ============================================================

# pd.read_csv() can read directly from a URL
# sep=";" — columns are separated by semicolons not commas
# header=1 — the real column names are on the second row
# encoding="latin-1" — DWD files use this encoding (not UTF-8)
df_temp = pd.read_csv(url, sep=";", header=1, encoding="latin-1")

# Quick first look — shape tells us (rows, columns)
print(f"Shape: {df_temp.shape}")
print(f"\nColumns: {df_temp.columns.tolist()}")
print(f"\nFirst few rows:")
df_temp.head()

Shape: (145, 20)

Columns: ['Jahr', 'Jahr.1', 'Brandenburg/Berlin', 'Brandenburg', 'Baden-Wuerttemberg', 'Bayern', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Niedersachsen/Hamburg/Bremen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Schleswig-Holstein', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Thueringen/Sachsen-Anhalt', 'Thueringen', 'Deutschland', 'Unnamed: 19']

First few rows:


,Jahr,Jahr.1,Brandenburg/Berlin,Brandenburg,Baden-Wuerttemberg,Bayern,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Niedersachsen/Hamburg/Bremen,Nordrhein-Westfalen,Rheinland-Pfalz,Schleswig-Holstein,Saarland,Sachsen,Sachsen-Anhalt,Thueringen/Sachsen-Anhalt,Thueringen,Deutschland,Unnamed: 19
0,1881,year,7.55,7.54,7.66,6.61,7.49,6.96,7.54,7.54,8.14,7.97,7.12,8.28,6.71,7.46,7.11,6.66,7.31,NaN
1,1882,year,8.99,8.97,8.08,7.33,8.25,8.54,8.88,8.88,9.03,8.55,8.78,8.79,8.12,8.81,8.35,7.77,8.34,NaN
2,1883,year,8.42,8.41,7.77,6.85,7.96,7.95,8.39,8.39,8.71,8.26,8.18,8.51,7.46,8.32,7.87,7.31,7.88,NaN
3,1884,year,9.11,9.10,8.44,7.52,8.58,8.73,9.09,9.10,9.39,8.94,8.86,9.18,8.21,8.94,8.47,7.89,8.57,NaN
4,1885,year,8.40,8.39,7.82,7.04,7.66,7.68,7.94,7.94,8.31,8.01,7.62,8.30,7.73,8.07,7.67,7.16,7.74,NaN


In [5]:
# Print just the last 3 column names to confirm the exact name
print(df_temp.columns.tolist()[-3:])

['Thueringen', 'Deutschland', 'Unnamed: 19']


In [9]:
df_temp.drop(columns = ['Jahr.1', 'Unnamed: 19'], inplace=True)
df_temp

,Jahr,Brandenburg/Berlin,Brandenburg,Baden-Wuerttemberg,Bayern,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Niedersachsen/Hamburg/Bremen,Nordrhein-Westfalen,Rheinland-Pfalz,Schleswig-Holstein,Saarland,Sachsen,Sachsen-Anhalt,Thueringen/Sachsen-Anhalt,Thueringen,Deutschland
0,1881,7.55,7.54,7.66,6.61,7.49,6.96,7.54,7.54,8.14,7.97,7.12,8.28,6.71,7.46,7.11,6.66,7.31
1,1882,8.99,8.97,8.08,7.33,8.25,8.54,8.88,8.88,9.03,8.55,8.78,8.79,8.12,8.81,8.35,7.77,8.34
2,1883,8.42,8.41,7.77,6.85,7.96,7.95,8.39,8.39,8.71,8.26,8.18,8.51,7.46,8.32,7.87,7.31,7.88
3,1884,9.11,9.10,8.44,7.52,8.58,8.73,9.09,9.10,9.39,8.94,8.86,9.18,8.21,8.94,8.47,7.89,8.57
4,1885,8.40,8.39,7.82,7.04,7.66,7.68,7.94,7.94,8.31,8.01,7.62,8.30,7.73,8.07,7.67,7.16,7.74
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140,2021,9.67,9.66,8.83,8.31,8.98,9.45,9.69,9.69,9.79,9.45,9.50,9.72,8.86,9.63,9.12,8.45,9.16
141,2022,10.76,10.75,10.59,9.90,10.65,10.21,10.78,10.78,11.21,11.18,10.25,11.52,10.17,10.84,10.47,9.97,10.52
142,2023,10.89,10.88,10.69,10.06,10.67,10.32,10.89,10.89,11.23,11.12,10.29,11.38,10.44,11.02,10.63,10.13,10.63
143,2024,11.54,11.53,10.56,10.27,10.74,10.96,11.26,11.26,11.32,10.94,10.77,11.09,10.95,11.48,11.03,10.45,10.89


**Note:** Some Bundesländer appear as combined regions (e.g. Brandenburg/Berlin).  
For national trend analysis we use the `Deutschland` column.  
Brandenburg will be extracted separately for the Berlin/wildfire analysis in RQ5.

In [10]:
# Save to raw/dwd/ folder
# index=False means don't write the row numbers as a column
df_temp.to_csv(DWD_RAW / "germany_annual_temp_1881_2025.csv", index=False)

print("Saved ✓")

Saved ✓


**Note:** .csv file there. All good. Clear increase in avg temp as expected. We will focus on Deutschland column.
Temperature anomalies will be calculated in notebook 01 using 1961–1990 as baseline period (international standard). Result will be cross-checked against DWD's official anomaly series.

### 1.2 DWD — Hot Days (Heiße Tage) per year
Days per year where maximum temperature exceeded 35°C.  
Available from 1951. Key indicator for RQ1 (trend) and RQ5 (Berlin heat stress).

In [4]:
# ============================================================
# DWD — Annual number of hot days for Germany (1951–present)
# ============================================================


url_hotdays = (
    "https://opendata.dwd.de/climate_environment/CDC/regional_averages_DE/annual/hot_days/regional_averages_txbs_year.txt"
)

# Send a request to that URL and get the response
response_hotdays = requests.get(url_hotdays)

# Status code 200 means the server responded with "OK"
# Anything else means something went wrong
print(f"Status code: {response_hotdays.status_code}")

# Show the first 1500 characters of the raw file
# so we can see how DWD formats it before we load it properly
print("\n--- RAW FILE PREVIEW ---\n")
print(response_hotdays.text[:1500])

Status code: 200

--- RAW FILE PREVIEW ---

Zeitreihen fuer Gebietsmittel fuer Bundeslaender und Kombinationen von Bundeslaender, erstellt am: 20260120
Jahr;Jahr;Brandenburg/Berlin;Brandenburg;Baden-Wuerttemberg;Bayern;Hessen;Mecklenburg-Vorpommern;Niedersachsen;Niedersachsen/Hamburg/Bremen;Nordrhein-Westfalen;Rheinland-Pfalz;Schleswig-Holstein;Saarland;Sachsen;Sachsen-Anhalt;Thueringen/Sachsen-Anhalt;Thueringen;Deutschland;
1951;year;     5.15;     5.13;     3.52;     3.47;     3.42;     2.21;     1.71;     1.71;     1.73;     1.97;     0.38;     1.38;     4.67;     4.25;     4.05;     3.79;     3.02;
1952;year;     5.46;     5.48;    14.31;    12.27;    10.37;     1.64;     2.55;     2.52;     6.71;    10.95;     0.82;    11.69;     8.81;     5.57;     6.64;     8.03;     7.91;
1953;year;    10.51;    10.43;     4.93;     3.34;     6.44;     3.91;     4.08;     4.09;     3.73;     6.21;     1.88;     5.93;     6.98;     7.30;     6.27;     4.93;     5.08;
1954;year;     5.16;     5.1

In [6]:
df_hotdays = pd.read_csv(url_hotdays, sep=";", header=1, encoding="latin-1")

# Quick first look — shape tells us (rows, columns)
print(f"Shape: {df_hotdays.shape}")
print(f"\nColumns: {df_hotdays.columns.tolist()}")
print(f"\nFirst few rows:")
df_hotdays.head()

Shape: (75, 20)

Columns: ['Jahr', 'Jahr.1', 'Brandenburg/Berlin', 'Brandenburg', 'Baden-Wuerttemberg', 'Bayern', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Niedersachsen/Hamburg/Bremen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Schleswig-Holstein', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Thueringen/Sachsen-Anhalt', 'Thueringen', 'Deutschland', 'Unnamed: 19']

First few rows:


,Jahr,Jahr.1,Brandenburg/Berlin,Brandenburg,Baden-Wuerttemberg,Bayern,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Niedersachsen/Hamburg/Bremen,Nordrhein-Westfalen,Rheinland-Pfalz,Schleswig-Holstein,Saarland,Sachsen,Sachsen-Anhalt,Thueringen/Sachsen-Anhalt,Thueringen,Deutschland,Unnamed: 19
0,1951,year,5.15,5.13,3.52,3.47,3.42,2.21,1.71,1.71,1.73,1.97,0.38,1.38,4.67,4.25,4.05,3.79,3.02,NaN
1,1952,year,5.46,5.48,14.31,12.27,10.37,1.64,2.55,2.52,6.71,10.95,0.82,11.69,8.81,5.57,6.64,8.03,7.91,NaN
2,1953,year,10.51,10.43,4.93,3.34,6.44,3.91,4.08,4.09,3.73,6.21,1.88,5.93,6.98,7.30,6.27,4.93,5.08,NaN
3,1954,year,5.16,5.14,2.43,1.83,2.98,1.88,1.52,1.51,1.37,3.02,0.69,3.03,4.11,4.57,4.05,3.37,2.53,NaN
4,1955,year,1.14,1.12,1.19,0.72,1.54,0.24,0.48,0.47,1.21,1.50,0.02,1.96,1.14,1.27,1.27,1.26,0.93,NaN


In [7]:
df_hotdays.drop(columns = ['Jahr.1', 'Unnamed: 19'], inplace=True)
df_hotdays

,Jahr,Brandenburg/Berlin,Brandenburg,Baden-Wuerttemberg,Bayern,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Niedersachsen/Hamburg/Bremen,Nordrhein-Westfalen,Rheinland-Pfalz,Schleswig-Holstein,Saarland,Sachsen,Sachsen-Anhalt,Thueringen/Sachsen-Anhalt,Thueringen,Deutschland
0,1951,5.15,5.13,3.52,3.47,3.42,2.21,1.71,1.71,1.73,1.97,0.38,1.38,4.67,4.25,4.05,3.79,3.02
1,1952,5.46,5.48,14.31,12.27,10.37,1.64,2.55,2.52,6.71,10.95,0.82,11.69,8.81,5.57,6.64,8.03,7.91
2,1953,10.51,10.43,4.93,3.34,6.44,3.91,4.08,4.09,3.73,6.21,1.88,5.93,6.98,7.30,6.27,4.93,5.08
3,1954,5.16,5.14,2.43,1.83,2.98,1.88,1.52,1.51,1.37,3.02,0.69,3.03,4.11,4.57,4.05,3.37,2.53
4,1955,1.14,1.12,1.19,0.72,1.54,0.24,0.48,0.47,1.21,1.50,0.02,1.96,1.14,1.27,1.27,1.26,0.93
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,2021,9.28,9.19,4.81,4.64,4.07,4.72,2.86,2.86,2.92,3.98,1.47,3.88,5.10,6.70,5.27,3.41,4.53
71,2022,19.76,19.73,21.69,15.90,20.55,12.18,15.51,15.43,17.67,22.53,6.99,29.75,16.23,20.43,18.34,15.62,17.30
72,2023,13.31,13.30,18.36,15.48,10.89,5.37,7.11,7.09,8.83,11.95,1.97,13.55,12.72,11.67,10.62,9.25,11.45
73,2024,19.63,19.62,14.66,13.39,11.78,9.66,7.43,7.41,8.31,11.49,3.47,11.58,19.29,18.23,16.22,13.60,12.47


In [8]:
# Save to raw/dwd/ folder
# index=False means don't write the row numbers as a column
df_hotdays.to_csv(DWD_RAW / "germany_hot_days_1951_2025.csv", index=False)

print("Saved ✓")

Saved ✓


**Note:** .csv file there. All good. Clear increase in number of hot days as expected. Would be interesting to check which state has the most hot days.

### 1.3 DWD — Hot Days (Heiße Tage) per year
Days per year where maximum temperature exceeded 35°C.  
Available from 1951. Key indicator for RQ1 (trend) and RQ5 (Berlin heat stress).

### 1.3 DWD — Downloading remaining data sets with a loop
- precipitation — total annual rainfall
- heavy rainfall days — days with precipitation >= 10mm
- frost days — days where minimum temperature dropped below 0°C
- tropical nights — nights where minimum temperature stayed above 20°C

In [6]:
dwd_remaining_files = [
    ("https://opendata.dwd.de/climate_environment/CDC/regional_averages_DE/annual/precipitation/regional_averages_rr_year.txt", "germany_total_precipitation.csv"),
    ("https://opendata.dwd.de/climate_environment/CDC/regional_averages_DE/annual/precipGE10mm_days/regional_averages_rrsfs_year.txt", "germany_heavy_rainfall_days.csv"),
    ("https://opendata.dwd.de/climate_environment/CDC/regional_averages_DE/annual/frost_days/regional_averages_tnas_year.txt", "germany_frost_days.csv"),
    ("https://opendata.dwd.de/climate_environment/CDC/regional_averages_DE/annual/tropical_nights_tminGE20/regional_averages_tnes_year.txt", "germany_tropical_nights.csv")
]

for url, filename in dwd_remaining_files:
    print(f"Trying: {url}")
    df_dwd = pd.read_csv(url, sep=";", header=1, encoding="latin-1")
    df_dwd.drop(columns = ['Jahr.1', 'Unnamed: 19'], inplace=True)
    df_dwd.to_csv(DWD_RAW / filename, index=False)
    # download, load, drop columns, save



Trying: https://opendata.dwd.de/climate_environment/CDC/regional_averages_DE/annual/precipitation/regional_averages_rr_year.txt
Trying: https://opendata.dwd.de/climate_environment/CDC/regional_averages_DE/annual/precipGE10mm_days/regional_averages_rrsfs_year.txt
Trying: https://opendata.dwd.de/climate_environment/CDC/regional_averages_DE/annual/frost_days/regional_averages_tnas_year.txt
Trying: https://opendata.dwd.de/climate_environment/CDC/regional_averages_DE/annual/tropical_nights_tminGE20/regional_averages_tnes_year.txt


In [11]:
df_total_precipitation = pd.read_csv(DWD_RAW / "germany_total_precipitation.csv")
df_frost_days      = pd.read_csv(DWD_RAW / "germany_frost_days.csv")
df_heavy_rainfall  = pd.read_csv(DWD_RAW / "germany_heavy_rainfall_days.csv")
df_tropical_nights = pd.read_csv(DWD_RAW / "germany_tropical_nights.csv")


In [12]:
df_total_precipitation.head()

,Jahr,Brandenburg/Berlin,Brandenburg,Baden-Wuerttemberg,Bayern,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Niedersachsen/Hamburg/Bremen,Nordrhein-Westfalen,Rheinland-Pfalz,Schleswig-Holstein,Saarland,Sachsen,Sachsen-Anhalt,Thueringen/Sachsen-Anhalt,Thueringen,Deutschland
0,1881,521.3,521.0,828.6,804.9,649.5,494.5,643.5,643.4,792.1,711.2,675.7,776.2,731.0,536.2,588.9,655.0,693.5
1,1882,673.6,673.0,1255.2,1085.3,1005.1,568.5,758.8,757.3,1032.1,1078.9,785.4,1210.3,952.3,666.8,762.3,882.4,926.7
2,1883,500.6,500.3,870.5,832.4,656.8,518.9,620.4,619.9,754.6,713.6,684.6,819.5,655.9,474.9,527.2,593.0,687.2
3,1884,635.4,634.3,776.4,797.2,699.4,600.1,755.2,754.6,808.3,700.3,767.4,769.0,723.8,635.9,652.0,672.2,732.2
4,1885,550.2,549.7,990.9,830.2,691.8,537.9,646.7,646.1,734.6,793.3,684.6,925.1,665.0,536.3,571.5,615.7,718.4


In [13]:
df_frost_days.head()

,Jahr,Brandenburg/Berlin,Brandenburg,Baden-Wuerttemberg,Bayern,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Niedersachsen/Hamburg/Bremen,Nordrhein-Westfalen,Rheinland-Pfalz,Schleswig-Holstein,Saarland,Sachsen,Sachsen-Anhalt,Thueringen/Sachsen-Anhalt,Thueringen,Deutschland
0,1951,79.45,79.79,99.53,117.17,81.24,74.85,71.65,71.51,56.23,70.74,63.55,65.41,87.59,80.79,88.72,99.08,85.57
1,1952,110.82,110.91,117.32,127.31,105.71,105.55,98.45,98.32,86.70,99.27,95.43,97.67,118.65,110.05,113.92,118.98,109.43
2,1953,91.13,91.30,117.09,126.08,92.66,84.26,73.49,73.31,68.27,87.28,65.55,83.74,94.22,83.56,91.33,101.48,94.54
3,1954,98.99,99.32,105.27,116.72,91.58,95.34,84.46,84.37,71.62,84.88,86.07,78.54,103.14,93.93,99.92,107.73,96.70
4,1955,112.43,112.57,128.30,134.60,121.86,111.79,104.62,104.56,100.34,118.47,101.55,112.47,118.07,108.60,115.71,124.98,117.22


In [14]:
df_heavy_rainfall.head()

,Jahr,Brandenburg/Berlin,Brandenburg,Baden-Wuerttemberg,Bayern,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Niedersachsen/Hamburg/Bremen,Nordrhein-Westfalen,Rheinland-Pfalz,Schleswig-Holstein,Saarland,Sachsen,Sachsen-Anhalt,Thueringen/Sachsen-Anhalt,Thueringen,Deutschland
0,1951,12.63,12.63,29.12,23.88,19.54,15.75,20.49,20.55,25.35,24.03,26.08,29.07,14.19,13.43,15.27,17.68,21.11
1,1952,10.32,10.36,33.99,30.45,22.72,11.74,16.94,16.87,23.55,26.07,18.39,33.04,17.56,14.05,17.16,21.23,22.14
2,1953,9.75,9.74,21.70,19.37,13.06,10.37,13.21,13.18,16.06,12.53,16.95,11.36,14.11,9.47,11.29,13.67,15.03
3,1954,14.69,14.67,30.54,29.61,21.04,15.68,23.04,23.08,27.95,20.93,25.20,24.10,23.22,16.11,17.75,19.89,23.71
4,1955,13.39,13.41,29.11,28.82,19.16,13.37,17.20,17.18,21.29,18.75,14.96,25.22,19.83,14.89,16.76,19.21,20.77


In [15]:
df_tropical_nights.head()

,Jahr,Brandenburg/Berlin,Brandenburg,Baden-Wuerttemberg,Bayern,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Niedersachsen/Hamburg/Bremen,Nordrhein-Westfalen,Rheinland-Pfalz,Schleswig-Holstein,Saarland,Sachsen,Sachsen-Anhalt,Thueringen/Sachsen-Anhalt,Thueringen,Deutschland
0,1951,0.02,0.02,0.00,0.00,0.03,0.00,0.00,0.00,0.00,0.01,0.00,0.00,0.14,0.01,0.01,0.01,0.01
1,1952,0.20,0.21,1.24,0.74,1.74,0.00,0.03,0.03,1.02,2.69,0.00,3.62,1.00,0.18,0.25,0.34,0.75
2,1953,0.02,0.00,0.03,0.00,0.05,0.02,0.00,0.00,0.02,0.01,0.01,0.08,0.00,0.00,0.00,0.00,0.01
3,1954,0.01,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.01,0.00,0.00,0.08,0.02,0.03,0.02,0.00,0.01
4,1955,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.01,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


## 2. EM Dat — Emergency Events Data Base

Maintained by the Centre for Research on the Epidemiology of Disasters (CRED) at the University of Louvain and is open access for non-commercial use. It covers natural disasters from 1900 to present with economic loss data by country and disaster type. You can filter for Germany specifically.

In [17]:
EMDAT_RAW = PROJECT_ROOT / "data" / "raw" / "emdat"
EMDAT_RAW.mkdir(parents=True, exist_ok=True)

In [19]:
import shutil
shutil.copy(
    Path.home() / "Downloads" / "public_emdat_custom_request_2026-05-20_50b36aee-0d44-494b-a169-ce878de2fef8.xlsx",
    EMDAT_RAW / "emdat_germany_climate_disasters_2002_2024.xlsx"
)
print("Saved ✓")

Saved ✓


In [20]:
df_emdat = pd.read_excel(EMDAT_RAW / "emdat_germany_climate_disasters_2002_2024.xlsx")
print(df_emdat.shape)
df_emdat.head(3)

(61, 47)


,DisNo.,Historic,Classification Key,Disaster Group,Disaster Subgroup,Disaster Type,Disaster Subtype,External IDs,Event Name,ISO,Country,Subregion,Region,Location,Origin,Associated Types,OFDA/BHA Response,Appeal,Declaration,AID Contribution ('000 US$),Magnitude,Magnitude Scale,Latitude,Longitude,River Basin,Start Year,Start Month,Start Day,End Year,End Month,End Day,Total Deaths,No. Injured,No. Affected,No. Homeless,Total Affected,Reconstruction Costs ('000 US$),"Reconstruction Costs, Adjusted ('000 US$)",Insured Damage ('000 US$),"Insured Damage, Adjusted ('000 US$)",Total Damage ('000 US$),"Total Damage, Adjusted ('000 US$)",CPI,Admin Units,GADM Admin Units,Entry Date,Last Update
0,2024-0369-DEU,No,nat-met-ext-hea,Natural,Meteorological,Extreme temperature,Heat wave,NaN,NaN,DEU,Germany,Western Europe,Europe,NaN,NaN,NaN,No,No,No,NaN,NaN,°C,NaN,NaN,NaN,2024,6,1.0,2024,9,30.0,6282.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,97.436176,NaN,NaN,2025-09-29,2025-09-30
1,2017-0310-DEU,No,nat-hyd-flo-riv,Natural,Hydrological,Flood,Riverine flood,HANZE:6002,NaN,DEU,Germany,Western Europe,Europe,"Hildesheim in Niedersachsen, districts surroun...",Low pressure Alfred,NaN,No,No,No,NaN,NaN,Km2,NaN,NaN,Gose and Abzucht rivers,2017,7,24.0,2017,7,27.0,1.0,NaN,600.0,NaN,600.0,NaN,NaN,NaN,NaN,NaN,NaN,76.137604,"[{""adm1_code"":1316,""adm1_name"":""Niedersachsen""...","[{""gid_1"":""DEU.16_1"",""migration_date"":""2025-12...",2017-08-02,2025-12-20
2,2022-0465-DEU,No,nat-met-ext-hea,Natural,Meteorological,Extreme temperature,Heat wave,GLIDE:HT-2022-000399,NaN,DEU,Germany,Western Europe,Europe,NaN,NaN,NaN,No,No,No,NaN,40.3,°C,NaN,NaN,NaN,2022,5,30.0,2022,9,4.0,8173.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,90.902697,NaN,NaN,2022-07-28,2025-06-25


## 3. Destatis — Bruttowertschöpfung nach Wirtschaftsbereichen

Source: Fachserie 18 Reihe 1.4 — Detaillierte Jahresergebnisse  
Sheet used: 2.2.1 — Bruttowertschöpfung in jeweiligen Preisen (current prices)  
Coverage: 1991–2024, all major economic sectors

**Sectors available:**
- Land- und Forstwirtschaft, Fischerei (Agriculture)
- Produzierendes Gewerbe ohne Baugewerbe (Manufacturing)
- Baugewerbe (Construction)
- Handel, Verkehr, Gastgewerbe (Trade, Transport, Hospitality)
- Information und Kommunikation (IT)
- Finanz- und Versicherungsdienstleister (Finance)
- Grundstücks- und Wohnungswesen (Real Estate)
- Unternehmensdienstleister (Business Services)
- Öffentliche Dienstleister, Erziehung, Gesundheit (Public Services)
- Sonstige Dienstleister (Other Services)

**Note:** For climate damage correlation we focus on agriculture, 
manufacturing, and construction — most directly exposed to 
extreme weather events.

In [21]:
shutil.copy(
    Path.home() / "Downloads" / "inlandsprodukt-vorlaeufig-xlsx-2180140.xlsx",
    DESTATIS_RAW / "inlandsprodukt-vorlaeufig-xlsx-2180140.xlsx"
)
print("Saved ✓")

Saved ✓


In [22]:
# Load the Destatis sector GDP sheet
# header=None because the structure is complex — we saw it has multiple header rows
# We skip to row 6 where the actual column names start
df_destatis = pd.read_excel(
    DESTATIS_RAW / "inlandsprodukt-vorlaeufig-xlsx-2180140.xlsx",
    sheet_name="2.2.1",
    header=None,
    skiprows=6   # skip the title rows, start from column headers
)

print(f"Shape: {df_destatis.shape}")
df_destatis.head(10)

Shape: (123, 13)


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,Jahr,Insgesamt,"Land- und Forst-wirtschaft, Fischerei",Produzierendes Gewerbe ohne Baugewerbe,NaN,Bau-gewerbe,"Handel, Verkehr, Gast-gewerbe",Information und Kommuni-kation,Finanz- \nund Versiche-rungs-dienst-leister,Grund-\nstücks- \nund Wohnungs-wesen,Unter-nehmens-dienst-leister,"Öffentliche Dienst-leister, Erziehung, Gesundheit",Sonstige Dienst-leister
1,NaN,NaN,NaN,zusammen,darunter:\nVerarbei-\ntendes Gewerbe,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,1,2,3,4,5,6,7,8,9,10,11,12
3,NaN,Mrd. EUR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1991,1446.372,17.03,442.102,391.924,87.501,232.596,52.138,65.95,124.922,133.817,236.125,54.191
5,1992,1552.925,16.908,447.887,396.114,103.563,240.405,59.44,71.31,141.221,149.259,262.42,60.512
6,1993,1592.408,17.184,422.919,369.888,108.695,246.272,62.998,78.886,157.242,159.26,276.607,62.345
7,1994,1657.667,18.258,430.578,376.823,116.808,259.056,63.521,78.07,171.192,161.319,289.536,69.329
8,1995,1719.432,19.612,440.276,384.11,116.615,270.098,67.265,77.692,183.798,168.653,304.89,70.533
9,1996,1743.857,20.581,436.924,380.563,110.261,271.101,68.353,82.015,191.09,174.708,316.592,72.232


One thing to note for later — those first 4 rows (0-3) are header mess that I'll clean up properly in notebook 02. 

In [23]:
print(f"Destatis file loaded successfully ✓")
print(f"Shape: {df_destatis.shape}")
print(f"Years covered: 1991 to 2024")
print(f"Sectors: {df_destatis.shape[1] - 1} economic sectors")

Destatis file loaded successfully ✓
Shape: (123, 13)
Years covered: 1991 to 2024
Sectors: 12 economic sectors


## 4. GDV — Datenservice zum Naturgefahrenreport 2025
Source: Gesamtverband der Deutschen Versicherungswirtschaft (GDV)
Format: PDF — requires table extraction with pdfplumber
Coverage: Insured losses from natural hazards 1973–2024
Breakdown: Storm/hail, elemental damage (flood, heavy rain), 
           vehicles — by year and by insurance segment

**Key table we need:** Annual insured loss time series 1973–2024
This is our primary damage variable for RQ2 and RQ3.

In [24]:
# ============================================================
# GDV — Extract annual damage table from PDF
# ============================================================
# pdfplumber opens PDFs and can extract both text and tables
# The key table is on page 6 of the PDF (index 5 — zero-indexed)
# ============================================================

import pdfplumber

gdv_path = GDV_RAW / "gdv_naturgefahren_datenservice_2025.pdf"

with pdfplumber.open(gdv_path) as pdf:
    # Page 6 is index 5 (pdfplumber counts from 0)
    page = pdf.pages[5]
    
    # Extract all text from the page so we can see what we're working with
    print("=== PAGE 6 TEXT PREVIEW ===")
    print(page.extract_text()[:3000])

=== PAGE 6 TEXT PREVIEW ===
06 DATENSERVICE ZUM NGR 2025 ÜBERSICHT
Schäden durch Naturgefahren 2024 auf einen Blick
Schadenaufwand in der Sach- und Kraftfahrtversicherung
5,6 Mrd. €
Sachversicherung
in der Sach- und Kfz- Wohngebäude, Hausrat, Industrie, Kfz-Versicherung
Versicherung 2024 Gewerbe und Landwirtschaft Voll- und Teilkasko
4,4 1,15
1,8 Mrd. € 2,6 Mrd. € 1,05 Mrd. €
Mrd. € Mrd. €
803.000 255.000 290.000 12.200
Sturm- und Hagelschäden Elementarschäden Sturm- und Überschwemmungs-
Hagelschäden schäden
Quelle: GDV
Sachversicherung und Kfz: Schätzung Schadenaufwand Naturgefahren
Naturgefahren: Sturm/Hagel, ab 2002 auch Elementar; Kfz: Sturm, Hagel, Blitz und Überschwemmung; Schadenaufwand
hochgerechnet auf Bestand und Preise 2024 in Mrd. Euro
Sach davon: Sach davon: Sach Sach davon: Sach davon: Sach
Jahr Naturgefahren Sturm/Hagel Elementar Kfz Jahr Naturgefahren Sturm/Hagel Elementar Kfz
1973 3,4 3,4 0,3 1999 4,5 4,5 1,0
1974 2,1 2,1 0,5 2000 1,8 1,8 1,0
1975 1,4 1,4 0,2 2001 1,6 

In [25]:
# ============================================================
# GDV — Parse the damage table from page 6 text
# ============================================================
# The table is split into two columns in the PDF (1973-1998 left,
# 1999-2024 right). We extract all text and parse it with pandas.
# We use StringIO to read the text as if it were a file.
# ============================================================

from io import StringIO

with pdfplumber.open(gdv_path) as pdf:
    text = pdf.pages[5].extract_text()

# The table data starts after "Kfz" header line
# Let's find where the actual data rows start
lines = text.split('\n')
for i, line in enumerate(lines):
    print(f"{i}: {line}")

0: 06 DATENSERVICE ZUM NGR 2025 ÜBERSICHT
1: Schäden durch Naturgefahren 2024 auf einen Blick
2: Schadenaufwand in der Sach- und Kraftfahrtversicherung
3: 5,6 Mrd. €
4: Sachversicherung
5: in der Sach- und Kfz- Wohngebäude, Hausrat, Industrie, Kfz-Versicherung
6: Versicherung 2024 Gewerbe und Landwirtschaft Voll- und Teilkasko
7: 4,4 1,15
8: 1,8 Mrd. € 2,6 Mrd. € 1,05 Mrd. €
9: Mrd. € Mrd. €
10: 803.000 255.000 290.000 12.200
11: Sturm- und Hagelschäden Elementarschäden Sturm- und Überschwemmungs-
12: Hagelschäden schäden
13: Quelle: GDV
14: Sachversicherung und Kfz: Schätzung Schadenaufwand Naturgefahren
15: Naturgefahren: Sturm/Hagel, ab 2002 auch Elementar; Kfz: Sturm, Hagel, Blitz und Überschwemmung; Schadenaufwand
16: hochgerechnet auf Bestand und Preise 2024 in Mrd. Euro
17: Sach davon: Sach davon: Sach Sach davon: Sach davon: Sach
18: Jahr Naturgefahren Sturm/Hagel Elementar Kfz Jahr Naturgefahren Sturm/Hagel Elementar Kfz
19: 1973 3,4 3,4 0,3 1999 4,5 4,5 1,0
20: 1974 2,1 2,1 0

In [26]:
# ============================================================
# Parse the two-column table structure
# Each line has: Jahr Value Value Value Kfz | Jahr Value Value Value Kfz
# We split each line into left year and right year
# ============================================================

import re

data_lines = lines[19:45]  # rows with actual year data

rows = []

for line in data_lines:
    # Skip footer lines
    if 'vorläufig' in line or 'GDV' in line or 'VDG' in line or '©' in line:
        continue
    
    # Extract all numbers from the line
    # Pattern: year followed by 2-4 decimal values
    numbers = re.findall(r'\d+(?:,\d+)?', line)
    
    # Convert German decimals (comma) to float
    numbers = [float(n.replace(',', '.')) for n in numbers]
    
    print(f"Line: '{line}' -> Numbers: {numbers}")

Line: '1973 3,4 3,4 0,3 1999 4,5 4,5 1,0' -> Numbers: [1973.0, 3.4, 3.4, 0.3, 1999.0, 4.5, 4.5, 1.0]
Line: '1974 2,1 2,1 0,5 2000 1,8 1,8 1,0' -> Numbers: [1974.0, 2.1, 2.1, 0.5, 2000.0, 1.8, 1.8, 1.0]
Line: '1975 1,4 1,4 0,2 2001 1,6 1,6 0,7' -> Numbers: [1975.0, 1.4, 1.4, 0.2, 2001.0, 1.6, 1.6, 0.7]
Line: '1976 9,1 9,1 0,5 2002 13,3 5,9 7,4 1,9' -> Numbers: [1976.0, 9.1, 9.1, 0.5, 2002.0, 13.3, 5.9, 7.4, 1.9]
Line: '1977 2,0 2,0 0,2 2003 2,6 2,1 0,5 1,2' -> Numbers: [1977.0, 2.0, 2.0, 0.2, 2003.0, 2.6, 2.1, 0.5, 1.2]
Line: '1978 1,5 1,5 0,2 2004 2,8 2,3 0,4 0,7' -> Numbers: [1978.0, 1.5, 1.5, 0.2, 2004.0, 2.8, 2.3, 0.4, 0.7]
Line: '1979 1,6 1,6 0,2 2005 3,0 2,2 0,9 0,7' -> Numbers: [1979.0, 1.6, 1.6, 0.2, 2005.0, 3.0, 2.2, 0.9, 0.7]
Line: '1980 1,4 1,4 0,2 2006 3,2 2,3 0,9 0,9' -> Numbers: [1980.0, 1.4, 1.4, 0.2, 2006.0, 3.2, 2.3, 0.9, 0.9]
Line: '1981 2,2 2,2 0,5 2007 8,0 7,1 1,0 1,1' -> Numbers: [1981.0, 2.2, 2.2, 0.5, 2007.0, 8.0, 7.1, 1.0, 1.1]
Line: '1982 2,3 2,3 0,3 2008 4,2 3,

In [27]:
# ============================================================
# Build the GDV damage DataFrame
# ============================================================
# Structure per line: left_year, sach, sturm_hagel, [elementar], kfz,
#                     right_year, sach, sturm_hagel, [elementar], kfz
# Before 2002: no Elementar column (only 3 values after year)
# From 2002: Elementar added (4 values after year)
# ============================================================

rows = []

for line in data_lines:
    if 'vorläufig' in line or 'GDV' in line or 'VDG' in line or '©' in line:
        continue
    
    numbers = re.findall(r'\d+(?:,\d+)?', line)
    numbers = [float(n.replace(',', '.')) for n in numbers]
    
    # Fix the superscript issue: 20231 → 2023, 20241 → 2024
    def fix_year(y):
        y = int(y)
        if y == 20231:
            return 2023
        if y == 20241:
            return 2024
        return y
    
    # LEFT SIDE of the table
    left_year = fix_year(numbers[0])
    
    if left_year < 2002:
        # Format: year, sach_total, sturm_hagel, kfz (3 values)
        rows.append({
            'Jahr': left_year,
            'sach_naturgefahren_mrd': numbers[1],
            'sach_sturm_hagel_mrd': numbers[2],
            'sach_elementar_mrd': None,
            'kfz_mrd': numbers[3]
        })
        # RIGHT SIDE starts at index 4
        right_start = 4
    else:
        # Format: year, sach_total, sturm_hagel, elementar, kfz (4 values)
        rows.append({
            'Jahr': left_year,
            'sach_naturgefahren_mrd': numbers[1],
            'sach_sturm_hagel_mrd': numbers[2],
            'sach_elementar_mrd': numbers[3],
            'kfz_mrd': numbers[4]
        })
        right_start = 5
    
    # RIGHT SIDE of the table
    if right_start < len(numbers):
        right_year = fix_year(numbers[right_start])
        
        if right_year < 2002:
            rows.append({
                'Jahr': right_year,
                'sach_naturgefahren_mrd': numbers[right_start + 1],
                'sach_sturm_hagel_mrd': numbers[right_start + 2],
                'sach_elementar_mrd': None,
                'kfz_mrd': numbers[right_start + 3]
            })
        else:
            rows.append({
                'Jahr': right_year,
                'sach_naturgefahren_mrd': numbers[right_start + 1],
                'sach_sturm_hagel_mrd': numbers[right_start + 2],
                'sach_elementar_mrd': numbers[right_start + 3],
                'kfz_mrd': numbers[right_start + 4]
            })

# Build DataFrame and sort by year
df_gdv = pd.DataFrame(rows).sort_values('Jahr').reset_index(drop=True)

print(f"Shape: {df_gdv.shape}")
print(f"Years: {df_gdv['Jahr'].min():.0f} to {df_gdv['Jahr'].max():.0f}")
print(f"\nFirst few rows:")
print(df_gdv.head())
print(f"\nLast few rows:")
print(df_gdv.tail())

Shape: (52, 5)
Years: 1973 to 2024

First few rows:
   Jahr  sach_naturgefahren_mrd  sach_sturm_hagel_mrd  sach_elementar_mrd  \
0  1973                     3.4                   3.4                 NaN   
1  1974                     2.1                   2.1                 NaN   
2  1975                     1.4                   1.4                 NaN   
3  1976                     9.1                   9.1                 NaN   
4  1977                     2.0                   2.0                 NaN   

   kfz_mrd  
0      0.3  
1      0.5  
2      0.2  
3      0.5  
4      0.2  

Last few rows:
    Jahr  sach_naturgefahren_mrd  sach_sturm_hagel_mrd  sach_elementar_mrd  \
47  2020                     2.4                   2.0                 0.4   
48  2021                    14.9                   2.4                12.6   
49  2022                     3.8                   3.5                 0.3   
50  2023                     4.1                   3.0                 1.1   
5

In [28]:
# Save to raw/gdv/
df_gdv.to_csv(GDV_RAW / "gdv_annual_damage_1973_2024.csv", index=False)
print("Saved ✓")

Saved ✓


Note the NaN values before 2002 for Elementar (that column only starts when elemental damage insurance became widespread), note 2021 as the clear outlier (Sturzflut Bernd), and note that all values are already inflation-adjusted to 2024 prices.